In [1]:
from fastapi import FastAPI, BackgroundTasks, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import RedirectResponse
from pydantic import BaseModel
from typing import List, Dict, Any, Optional
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
import os
import json
import yaml
import logging
import pandas as pd
import numpy as np

In [2]:
raw_path = "../src/ifrs9_framework/data/raw/synthetic_credit_data.parquet"

In [3]:
df = pd.read_parquet(raw_path)

In [4]:
df.head()

,cliente_id,idade,renda,uf,zona_risco_flag,bureau_score,obito_flag,codigo_contrato,tipo_produto,data_contratacao,prazo_meses,data_vencimento,limite_credito,valor_financiado,selic_contratacao,default_flag,churn_flag,data_evento
0,1,58.953998,2302.091770,MA,1,458.690273,0,4f2c04d0,Consignado SIAPE,2021-11-06,79,2028-05-03,0.0,10711.228896,5.081772,0,0,2028-05-03
1,1,58.953998,2302.091770,MA,1,458.690273,0,de9929d6,Consignado SIAPE,2022-06-12,93,2030-01-31,0.0,7319.579877,13.149033,0,0,2030-01-31
2,1,58.953998,2302.091770,MA,1,458.690273,0,ca147a05,Consignado INSS,2023-01-17,90,2030-06-09,0.0,5377.525954,12.454835,0,1,2024-09-08
3,2,50.064300,5547.355046,RJ,0,588.041646,0,f833304b,Consignado INSS,2023-01-26,95,2030-11-15,0.0,8629.994977,5.521695,0,0,2030-11-15
4,3,61.067640,3537.639538,MA,0,671.794326,0,5262dd9b,Consignado INSS,2022-06-09,84,2029-05-03,0.0,30517.265019,12.181277,0,1,2022-10-07


In [5]:
logger = logging.getLogger("MLOps-Orchestrator")
logger.setLevel(logging.INFO)
config = {
            'target_column': 'default_flag',
            'group_column': 'codigo_contrato',
            'test_size': 0.25,
            'random_state': 42
        }

def safe_col(df, col_name, default_val=0.0):
    if col_name in df.columns:
        return df[col_name]
    return pd.Series(default_val, index=df.index)

In [6]:
def features_engineer(df, config, logger):
    logger.info("Criando features comportamentais e financeiras...")
    df_eng = df.copy()
    
    # CONVERSÃO ABSOLUTA DE TODAS AS COLUNAS PARA STRING ANTES DE QUALQUER REPLACE
    cleaned_columns = []
    for c in df_eng.columns:
        col_str = str(c).replace("'", "").strip()
        cleaned_columns.append(col_str)
    df_eng.columns = cleaned_columns
    
    date_cols = ['DTA_INI_OPR', 'DTA_RFC', 'DTA_NAS']
    for col in date_cols:
        if col in df_eng.columns:
            df_eng[col] = pd.to_datetime(df_eng[col], errors='coerce')
            
    if 'IDT_EPC_BNF' in df_eng.columns:
        mapping_especie = config.get('mapping_especie', DOMAIN_MAPPINGS.get('mapping_especie', {}))
        df_eng['NEW_GRP_EPC_BNF'] = df_eng['IDT_EPC_BNF'].map(mapping_especie).fillna('OUTROS')
        
    if 'QTD_DIA_VCD' in df_eng.columns: df_eng['QTD_DIA_VCD'] = safe_col(df_eng, 'QTD_DIA_VCD', 0).clip(lower=0)
    if 'PZO_RMN' in df_eng.columns: df_eng['PZO_RMN'] = safe_col(df_eng, 'PZO_RMN', -1).clip(lower=-1)
    if 'TMP_RLN' in df_eng.columns: df_eng['TMP_RLN'] = safe_col(df_eng, 'TMP_RLN', 0).clip(lower=0)
    
    if 'TMP_CMO_BNF_DIA' in df_eng.columns:
        tmp_val = safe_col(df_eng, 'TMP_CMO_BNF_DIA', 0)
        df_eng['FLAG_TMP_CMO'] = np.where((tmp_val < 0) | (tmp_val > 20000), 1, 0)
        
    sld_ctb = safe_col(df_eng, 'SLD_CTB_LQD_PVSCLI', 0.0)
    vlr_ren = safe_col(df_eng, 'VLR_REN', 0.0)
    gra_vul = safe_col(df_eng, 'GRA_VUL', 0.0)
    idd_opr = safe_col(df_eng, 'IDD_DTA_OPR', 0.0)
    qtd_vcd = safe_col(df_eng, 'QTD_DIA_VCD', 0.0)
    qtd_pcl_vcd = safe_col(df_eng, 'QTD_PCL_VCD', 0.0)
    qtd_pcl_tot = safe_col(df_eng, 'QTD_PCL_TOT', 1.0)
    qtd_pcl_pag = safe_col(df_eng, 'QTD_PCL_PAG', 0.0)
    qtd_opr_vcd = safe_col(df_eng, 'QTD_OPR_VCD', 0.0)
    qtd_opr_tot = safe_col(df_eng, 'QTD_OPR_TOT', 1.0)
    tax_eft = safe_col(df_eng, 'TAX_EFT_ANO', 0.0)
    tmp_rln = safe_col(df_eng, 'TMP_RLN', 0.0)
    pzo_rmn = safe_col(df_eng, 'PZO_RMN', 1.0)

    df_eng['EXPOSICAO_SOBRE_RENDA'] = sld_ctb / (vlr_ren + 0.1)
    df_eng['RESILIENCIA_FINANCEIRA'] = gra_vul / (vlr_ren + 0.1)
    df_eng['MATURIDADE_FINANCEIRA'] = vlr_ren / (idd_opr.replace(0, 1) + 0.1)
    
    df_eng['DIAS_VENCIDOS_PARCELA'] = qtd_vcd / (qtd_pcl_vcd + 0.1)
    df_eng['PROP_PCL_VCD'] = qtd_pcl_vcd / qtd_pcl_tot.replace(0, 1)
    df_eng['PROP_PCL_PAG'] = qtd_pcl_pag / qtd_pcl_tot.replace(0, 1)
    df_eng['PROP_OPER_VCD'] = qtd_opr_vcd / qtd_opr_tot.replace(0, 1)
    
    df_eng['GRA_VUL_IDADE'] = gra_vul / (idd_opr + 0.1)
    df_eng['IDC_SUSTENTABILIDADE_ENCARGOS'] = vlr_ren / (tax_eft + 0.1)
    df_eng['CONFIABILIDADE'] = tmp_rln / (tax_eft + 0.1)
    df_eng['SALDO_POR_VUL'] = sld_ctb / (gra_vul + 0.1)
    df_eng['RENDA_SOBRE_PRAZO'] = vlr_ren / (pzo_rmn + 0.1)
    df_eng['RENDA_SOBRE_OPR_TOT'] = vlr_ren / (qtd_opr_tot + 0.1)
    
    idt_stu = safe_col(df_eng, 'IDT_STU_BNF', 0)
    df_eng['BENEFICIO_ATIVO'] = np.where(idt_stu == 3, 1, 0)
    df_eng['IDADA_QUAD'] = idd_opr ** 2
    df_eng['FLAG_SEM_ATRASO'] = np.where(qtd_vcd == 0, 1, 0)
    
    if 'TOTAL_DIAS_ATRASO_ACUMULADO' in df_eng.columns:
        tot_atraso = safe_col(df_eng, 'TOTAL_DIAS_ATRASO_ACUMULADO', 0.0)
        df_eng['MEDIA_DIAS_EM_ATRASO'] = tot_atraso / (qtd_pcl_pag + 0.1)
        
    if 'DTA_RFC' in df_eng.columns and 'DTA_NAS' in df_eng.columns:
        df_eng['FLAG_ANIVERSARIO'] = np.where(df_eng['DTA_RFC'].dt.month == df_eng['DTA_NAS'].dt.month, 1, 0)
        
    df_eng.replace([np.inf, -np.inf], np.nan, inplace=True)
    logger.info("Engenharia de features concluída com sucesso.")
    return df_eng

In [7]:
df_eng = features_engineer(df, config, logger)

In [8]:
df_eng.head()

,cliente_id,idade,renda,uf,zona_risco_flag,bureau_score,obito_flag,codigo_contrato,tipo_produto,data_contratacao,...,PROP_OPER_VCD,GRA_VUL_IDADE,IDC_SUSTENTABILIDADE_ENCARGOS,CONFIABILIDADE,SALDO_POR_VUL,RENDA_SOBRE_PRAZO,RENDA_SOBRE_OPR_TOT,BENEFICIO_ATIVO,IDADA_QUAD,FLAG_SEM_ATRASO
0,1,58.953998,2302.091770,MA,1,458.690273,0,4f2c04d0,Consignado SIAPE,2021-11-06,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,1
1,1,58.953998,2302.091770,MA,1,458.690273,0,de9929d6,Consignado SIAPE,2022-06-12,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,1
2,1,58.953998,2302.091770,MA,1,458.690273,0,ca147a05,Consignado INSS,2023-01-17,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,1
3,2,50.064300,5547.355046,RJ,0,588.041646,0,f833304b,Consignado INSS,2023-01-26,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,1
4,3,61.067640,3537.639538,MA,0,671.794326,0,5262dd9b,Consignado INSS,2022-06-09,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,1


In [9]:
def div_train_test_split(df, config):
    """
    1. DIVISÃO COM DATAS PRESERVADAS (USANDO GROUP SPLIT)
    Garante que o mesmo grupo (ex: COD_OPR_ATV) não apareça no treino e teste.
    """
    target = config.get('target_column', 'default_flag')
    group_col = config.get('group_column', 'codigo_contrato')
    
    # Validação de segurança
    if group_col not in df.columns:
        raise ValueError(f"Coluna de agrupamento '{group_col}' não encontrada!")
        
    logger.info(f"Realizando Split Treino/Teste por Grupos de contrato ({group_col})...")
    
    # Split Treino+Val vs Teste
    splitter = GroupShuffleSplit(n_splits=1, test_size=config.get('test_size', 0.25), random_state=config.get('random_state', 42))
    train_val_idx, test_idx = next(splitter.split(df, df[target], groups=df[group_col]))
    
    df_train_val = df.iloc[train_val_idx].copy()
    df_test = df.iloc[test_idx].copy()
    
    logger.info(f"Shape Treino+Val: {df_train_val.shape}, Shape Teste: {df_test.shape}")
    
    return df_train_val, df_test

In [10]:
df_train_val, df_test = div_train_test_split(df_eng, config)

In [11]:
def build_preprocessor(df, config, logger):
    base_binary = config.get('binary_features', [])
    base_categorical = config.get('categorical_features', [])
    target = config.get('target_column', 'default_flag')
    
    binary_features = [f for f in base_binary if f in df.columns]
    categorical_features = [f for f in base_categorical if f in df.columns]
    
    num_cols = df.select_dtypes(include=np.number).columns
    numeric_features = [f for f in num_cols if f not in binary_features + categorical_features + [target]]
    
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    binary_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder())
    ])
    
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NAO_INFORMADO')),
        ('to_string', FunctionTransformer(converter_para_string, validate=False, feature_names_out='one-to-one'))
    ])
    
    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_transformer, numeric_features),
        ('bin', binary_transformer, binary_features),
        ('cat', categorical_transformer, categorical_features)
    ], remainder='drop')
    
    return preprocessor

In [14]:
def preparar_matrizes(df_train_val, df_test, config, preprocessor_func, tracker=None):
    """
    Separa X e Y, remove colunas constantes de forma vetorizada 
    e aplica o pré-processamento com captura rigorosa de exceções.
    """
    target = config.get('target_column', 'default_flag')
    group_col = config.get('group_column', 'COD_OPR_ATV')
    cols_to_drop_config = config.get('columns_to_drop', [])
    
    def log_and_track(msg):
        logger.info(msg)
        if tracker:
            tracker.update_node("step_2", "running", msg)

    def split_X_y_raw(df_subset):
        y_out = df_subset[target]
        cols_to_drop = [target] + [c for c in cols_to_drop_config if c in df_subset.columns]
        X_out = df_subset.drop(columns=cols_to_drop)
        return X_out, y_out
        
    log_and_track("Separando matrizes X e y...")
    X_train_val_raw, y_train_val = split_X_y_raw(df_train_val)
    X_test_raw, y_test = split_X_y_raw(df_test)
    
    log_and_track("Identificando colunas constantes (método vetorizado)...")
    nunique_vals = X_train_val_raw.nunique(dropna=False)
    cols_constantes = nunique_vals[nunique_vals <= 1].index.tolist()
    
    if cols_constantes:
        log_and_track(f"Removendo {len(cols_constantes)} colunas constantes: {cols_constantes}")
        X_train_val_raw.drop(columns=cols_constantes, inplace=True)
        X_test_raw.drop(columns=cols_constantes, inplace=True, errors='ignore')
        
    groups_train_val = df_train_val[group_col].values
    log_and_track(f"Realizando Split Treino/Validação por Grupos ({group_col})...")
    
    splitter_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=config.get('random_state', 42))
    tr_idx, val_idx = next(splitter_val.split(X_train_val_raw, y_train_val, groups=groups_train_val))
    
    X_train_raw = X_train_val_raw.iloc[tr_idx]
    y_train = y_train_val.iloc[tr_idx]
    X_val_raw = X_train_val_raw.iloc[val_idx]
    y_val = y_train_val.iloc[val_idx]
    
    log_and_track(f"Linhas finais -> Treino: {X_train_raw.shape[0]:,}, Validação: {X_val_raw.shape[0]:,}, Teste: {X_test_raw.shape[0]:,}")
    
    preprocessor = preprocessor_func(X_train_raw, config, logger)
    log_and_track("Aplicando pré-processamento nas matrizes (fit_transform)...")
    
    try:
        X_train_processed = preprocessor.fit_transform(X_train_raw)
        X_val_processed = preprocessor.transform(X_val_raw)
        X_test_processed = preprocessor.transform(X_test_raw)
        X_train_val_processed = preprocessor.transform(X_train_val_raw)
    except Exception as e:
        err_msg = f"CRASH NO PRÉ-PROCESSADOR: {str(e)}\n{traceback.format_exc()}"
        logger.error(err_msg)
        raise RuntimeError(err_msg)
        
    if hasattr(X_train_processed, 'columns'):
        all_cols = X_train_processed.columns.tolist()
    else:
        try:
            all_cols = list(preprocessor.get_feature_names_out())
        except Exception:
            all_cols = [f"Feature_{i}" for i in range(X_train_processed.shape[1])]
            
    base_cat = config.get('categorical_features', [])
    base_bin = config.get('binary_features', [])
    
    features_categoricas = [col for col in all_cols if any(c in col for c in base_cat)]
    features_binarias = [col for col in all_cols if any(b in col for b in base_bin)]
    features_numericas = [col for col in all_cols if col not in features_categoricas and col not in features_binarias]
    
    cat_indices = [all_cols.index(col) for col in features_categoricas]
    
    log_and_track(f"EXAME DE FEATURES | Total: {len(all_cols)} | Num: {len(features_numericas)} | Cat: {len(features_categoricas)}")
    
    groups_train = groups_train_val[tr_idx]
    groups_val = groups_train_val[val_idx]
    
    return {
        'preprocessor': preprocessor,
        'feature_names': all_cols,
        'cat_indices': cat_indices,
        'X_train_processed': X_train_processed, 'y_train': y_train,
        'X_val_processed': X_val_processed, 'y_val': y_val,
        'X_test_processed': X_test_processed, 'y_test': y_test,
        'X_train_val_processed': X_train_val_processed, 'y_train_val': y_train_val,
        'groups_train': groups_train, 'groups_val': groups_val
    }

def converter_para_string(x):
    return x.astype(str)

In [15]:
data_dict = preparar_matrizes(df_train_val, df_test, config, build_preprocessor)

In [16]:
data_dict

{'preprocessor': ColumnTransformer(transformers=[('num',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median')),
                                                  ('scaler', StandardScaler())]),
                                  ['cliente_id', 'idade', 'renda',
                                   'zona_risco_flag', 'bureau_score',
                                   'obito_flag', 'prazo_meses', 'limite_credito',
                                   'valor_financiado', 'selic_contratacao',
                                   'churn_flag']),
                                 ('bin',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='most_frequent')),
                                                  ('encoder',
                                                   OrdinalEncoder())]),
                                  [])